# EDA + extraction hiérarchique LUNA16 — pour MS-IPS et H-MS-IPS

Ce notebook produit **un seul jeu de features** exploitable par les deux méthodes
multi-patch (`ms-ips-luna16` et `h-ms-ips-luna16`).

- Grille alignée : zones grossières 128 px, patches fins 64 px (4 fins / zone).
- Encodeur : ResNet50 BYOL pré-entraîné (checkpoint fourni), features 2048-D.
- Sortie HDF5 par scan : `coarse (n_zones,2048)`, `fine (n_zones,4,2048)`,
  `coarse_yx`, `fine_yx`, `z`, attrs `label` / `n_zones`.

La mémoire ne tenant pas sur Kaggle, tout le scratch transite par `/kaggle/tmp`
et chaque split est nettoyé avant le suivant. HDF5 final -> `/kaggle/working`.

In [ ]:
# ============================================================
# STEP 1 — Environnement
# ============================================================
import os, glob, pickle, time, shutil
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import SimpleITK as sitk
import h5py
import copy
from collections import OrderedDict
from tqdm import tqdm
import multiprocessing as mp
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Device  : {DEVICE}")
print(f"PyTorch : {torch.__version__}")

LUNA_ROOT = '/kaggle/input/datasets/vafaeii/luna16'
SEG_DIR   = f'{LUNA_ROOT}/seg-lungs-LUNA16/seg-lungs-LUNA16'

# ── Scratch de session (pas de quota strict) ──────────────────
TMP_ROOT  = '/kaggle/tmp/luna16_hmsips'
OUT_DIR   = f'{TMP_ROOT}/coords'
CACHE_DIR = f'{TMP_ROOT}/cache'
# ── Livrables (poussés ensuite sur Kaggle comme dataset) ──────
HDF5_DIR  = '/kaggle/working/luna16_hmsips_features'
EDA_DIR   = '/kaggle/working/eda_luna16_multiscale'

for d in (OUT_DIR, CACHE_DIR, HDF5_DIR, EDA_DIR):
    os.makedirs(d, exist_ok=True)

# ── Checkpoint BYOL déjà entraîné (fourni) ────────────────────
CKPT_PATH = '/kaggle/input/datasets/tandem03/luna16-byol-outputs-version1/byol_epoch50.pth'
assert os.path.exists(CKPT_PATH), f"Checkpoint BYOL introuvable : {CKPT_PATH}"

# ── Grille hiérarchique alignée ───────────────────────────────
COARSE_SIZE = 128
FINE_SIZE   = 64
RATIO       = COARSE_SIZE // FINE_SIZE
N_FPZ       = RATIO * RATIO          # 4 patches fins / zone
FG_THRESH   = 0.01
TEST_FOLD   = 0
N_WORKER    = 4
assert COARSE_SIZE % FINE_SIZE == 0
print(f"COARSE_SIZE={COARSE_SIZE}  FINE_SIZE={FINE_SIZE}  n_fine_per_zone={N_FPZ}")

total, used, free = shutil.disk_usage('/kaggle/tmp')
print(f"/kaggle/tmp libre : {free/1e9:.1f} GB / {total/1e9:.1f} GB")

In [ ]:
# ============================================================
# STEP 2 — luna_utils
# ============================================================
def build_uid_index(luna_root=LUNA_ROOT, seg_dir=SEG_DIR):
    ann = pd.read_csv(os.path.join(luna_root, 'annotations.csv'))
    ann_uids = set(ann['seriesuid'].unique())
    seg_files = {os.path.basename(f).replace('.mhd', ''): f
                 for f in glob.glob(os.path.join(seg_dir, '*.mhd'))}
    index = OrderedDict()
    for i in range(10):
        pattern = os.path.join(luna_root, f'subset{i}', f'subset{i}', '*.mhd')
        for ct_path in sorted(glob.glob(pattern)):
            uid = os.path.basename(ct_path).replace('.mhd', '')
            index[uid] = {'ct': ct_path, 'seg': seg_files.get(uid),
                          'subset': i, 'label': 1 if uid in ann_uids else 0}
    return index

def load_ct_seg(info):
    ct  = sitk.ReadImage(info['ct'])
    seg = sitk.ReadImage(info['seg'])
    arr_ct  = sitk.GetArrayFromImage(ct).astype(np.float32)
    arr_seg = sitk.GetArrayFromImage(seg).astype(np.uint8)
    arr_norm = (np.clip(arr_ct, -1000, 400) + 1000) / 1400.0
    origin  = np.array(ct.GetOrigin())
    spacing = np.array(ct.GetSpacing())
    return arr_norm, arr_seg, origin, spacing

UID_INDEX = build_uid_index(LUNA_ROOT, SEG_DIR)
print(f"Scans indexés   : {len(UID_INDEX)}")
print(f"Avec masque seg : {sum(1 for v in UID_INDEX.values() if v['seg'])}")
print(f"Positifs        : {sum(v['label'] for v in UID_INDEX.values())}")

In [ ]:
# ============================================================
# STEP 3 — Encodeur BYOL (checkpoint fourni)
# ============================================================
import torchvision.models as models

class MLP(nn.Module):
    def __init__(self, in_dim=2048, hidden_dim=4096, out_dim=256):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(in_dim, hidden_dim), nn.BatchNorm1d(hidden_dim),
            nn.ReLU(inplace=True), nn.Linear(hidden_dim, out_dim))
    def forward(self, x): return self.net(x)

class EncoderWithProjector(nn.Module):
    def __init__(self):
        super().__init__()
        base = models.resnet50(weights=None)
        base.conv1 = nn.Conv2d(1, 64, kernel_size=7, stride=2, padding=3, bias=False)
        self.encoder   = nn.Sequential(*list(base.children())[:-1])
        self.projector = MLP(in_dim=2048, hidden_dim=4096, out_dim=256)
    def forward(self, x):
        h = self.encoder(x).flatten(start_dim=1)
        return h, self.projector(h)

class BYOLModel(nn.Module):
    def __init__(self, tau=0.996):
        super().__init__()
        self.tau = tau
        self.online    = EncoderWithProjector()
        self.predictor = MLP(in_dim=256, hidden_dim=4096, out_dim=256)
        self.target    = copy.deepcopy(self.online)
        for p in self.target.parameters(): p.requires_grad = False

ckpt  = torch.load(CKPT_PATH, map_location=DEVICE)
model = BYOLModel(tau=0.996).to(DEVICE)
model.load_state_dict(ckpt['model'])
print(f"Checkpoint BYOL chargé — epoch {ckpt['epoch']+1}, loss={ckpt['loss']:.4f}")

encoder = model.online.encoder.to(DEVICE)
encoder.eval()
for p in encoder.parameters(): p.requires_grad = False
print(f"Encodeur figé : {sum(p.numel() for p in encoder.parameters()):,} params")

In [ ]:
# ============================================================
# STEP 4 — EDA rapide (échantillon, mémoire-safe)
# Nombre de scans / labels + estimation zones & patches par scan.
# ============================================================
df = pd.DataFrame([{'uid': u, 'subset': v['subset'], 'label': v['label']}
                   for u, v in UID_INDEX.items()])
print("=== Répartition scans ===")
print(f"Total   : {len(df)}   pos={df['label'].sum()}   neg={(df['label']==0).sum()}")
print("\nPar subset :")
for i in range(10):
    sub = df[df['subset'] == i]
    print(f"  subset{i} : {len(sub):3d} scans | pos={sub['label'].sum():3d} "
          f"neg={(len(sub)-sub['label'].sum()):3d}")
print(f"\nTest fold = {TEST_FOLD} "
      f"(train={len(df[df['subset']!=TEST_FOLD])}, test={len(df[df['subset']==TEST_FOLD])})")

def count_zones_one(uid):
    _, arr_seg, _, _ = load_ct_seg(UID_INDEX[uid])
    Z, Y, X = arr_seg.shape
    nz = 0
    for z in range(Z):
        s = arr_seg[z]
        for yc in range(0, Y - COARSE_SIZE + 1, COARSE_SIZE):
            for xc in range(0, X - COARSE_SIZE + 1, COARSE_SIZE):
                if s[yc:yc+COARSE_SIZE, xc:xc+COARSE_SIZE].mean() >= FG_THRESH:
                    nz += 1
    return nz

sample_uids = list(UID_INDEX.keys())[:8]
zc = [count_zones_one(u) for u in sample_uids]
zc = np.array(zc)
print("\n=== Estimation par scan (échantillon 8) ===")
print(f"  Zones grossières / scan : min={zc.min()}  max={zc.max()}  moy={zc.mean():.0f}")
print(f"  Patches fins    / scan : moy={zc.mean()*N_FPZ:.0f}  (= zones × {N_FPZ})")
print(f"  Tokens MS-IPS   / scan : moy={zc.mean()*(N_FPZ+1):.0f}  (coarse + fine)")
print(f"\nEstimation globale ({len(df)} scans) :")
print(f"  ~{int(zc.mean()*len(df)):,} zones grossières")
print(f"  ~{int(zc.mean()*N_FPZ*len(df)):,} patches fins")

In [ ]:
# ============================================================
# STEP 5 — Extraction des coordonnées hiérarchiques (coarse + fine)
# ============================================================
def get_hierarchical_coords(arr_seg, coarse_size=COARSE_SIZE,
                            fine_size=FINE_SIZE, fg_thresh=FG_THRESH):
    Z, Y, X = arr_seg.shape
    ratio   = coarse_size // fine_size
    for z in range(Z):
        seg_slice = arr_seg[z]
        for yc in range(0, Y - coarse_size + 1, coarse_size):
            for xc in range(0, X - coarse_size + 1, coarse_size):
                if seg_slice[yc:yc+coarse_size, xc:xc+coarse_size].mean() < fg_thresh:
                    continue
                fine_list = []
                for fr in range(ratio):
                    for fc in range(ratio):
                        fine_list.append((yc + fr*fine_size, xc + fc*fine_size))
                yield z, yc, xc, fine_list

def get_hierarchical_coords_for_uid(uid):
    info = UID_INDEX[uid]
    try:
        _, arr_seg, _, _ = load_ct_seg(info)
        return uid, [(z, yc, xc, fl) for z, yc, xc, fl
                     in get_hierarchical_coords(arr_seg)]
    except Exception as e:
        print(f"ERR {uid[:40]} : {e}")
        return uid, []

def build_hierarchical_bounds(uids, subset_name):
    pool    = mp.Pool(N_WORKER)
    results = list(tqdm(pool.imap(get_hierarchical_coords_for_uid, uids),
                        total=len(uids)))
    pool.close(); pool.join()

    coarse_rows, fine_rows = [], []
    start_ids, end_ids, bound_names = [], [], []
    gid = 0
    for uid, rows in results:
        if len(rows) == 0:
            continue
        start_ids.append(gid); end_ids.append(gid + len(rows) - 1); bound_names.append(uid)
        for z, yc, xc, fine_list in rows:
            coarse_rows.append({'id_coarse': gid, 'name': uid,
                                'z': z, 'y_coarse': yc, 'x_coarse': xc})
            for fr, (yf, xf) in enumerate(fine_list):
                fine_rows.append({'id_coarse': gid, 'fine_rank': fr,
                                  'y_fine': yf, 'x_fine': xf})
            gid += 1

    coords_coarse_df = pd.DataFrame(coarse_rows)
    coords_fine_df   = pd.DataFrame(fine_rows)
    bounds_df        = pd.DataFrame({'names': bound_names,
                                     'start_id': start_ids, 'end_id': end_ids})

    with open(f'{OUT_DIR}/coords_coarse_{subset_name}.pkl', 'wb') as f: pickle.dump(coords_coarse_df, f)
    with open(f'{OUT_DIR}/coords_fine_{subset_name}.pkl', 'wb') as f: pickle.dump(coords_fine_df, f)
    with open(f'{OUT_DIR}/bounds_{subset_name}.pkl', 'wb') as f: pickle.dump(bounds_df, f)

    print(f"Zones grossières : {len(coords_coarse_df):,}")
    print(f"Patches fins     : {len(coords_fine_df):,}  (attendu {len(coords_coarse_df)*N_FPZ:,})")
    print(f"Scans            : {len(bounds_df)}")
    assert len(coords_fine_df) == len(coords_coarse_df) * N_FPZ
    return coords_coarse_df, coords_fine_df, bounds_df

train_uids = [u for u, v in UID_INDEX.items() if v['subset'] != TEST_FOLD]
test_uids  = [u for u, v in UID_INDEX.items() if v['subset'] == TEST_FOLD]
print(f"Train scans : {len(train_uids)}")
coords_coarse_train, coords_fine_train, bounds_train = \
    build_hierarchical_bounds(train_uids, f'train_fold{TEST_FOLD}')
print(f"\nTest scans : {len(test_uids)}")
coords_coarse_test, coords_fine_test, bounds_test = \
    build_hierarchical_bounds(test_uids, f'test_fold{TEST_FOLD}')

In [ ]:
# ============================================================
# STEP 6 — Cache memmap disque (coarse + fine) sur /kaggle/tmp
# ============================================================
def build_patch_cache(coords_coarse_df, coords_fine_df, cache_prefix):
    n_coarse = len(coords_coarse_df); n_fine = len(coords_fine_df)
    coarse_path = f'{cache_prefix}_coarse.npy'; fine_path = f'{cache_prefix}_fine.npy'

    if os.path.exists(coarse_path) and os.path.exists(fine_path):
        print(f"Cache déjà existant : {cache_prefix}_*")
        return (np.memmap(coarse_path, dtype=np.uint8, mode='r',
                          shape=(n_coarse, COARSE_SIZE, COARSE_SIZE)),
                np.memmap(fine_path, dtype=np.uint8, mode='r',
                          shape=(n_fine, FINE_SIZE, FINE_SIZE)))

    total, used, free = shutil.disk_usage('/kaggle/tmp')
    needed = (n_coarse*COARSE_SIZE**2 + n_fine*FINE_SIZE**2) / 1e9
    print(f"Création cache : {n_coarse:,} zones + {n_fine:,} fins  (~{needed:.1f} GB)")
    print(f"  /kaggle/tmp libre : {free/1e9:.1f} GB")
    assert free/1e9 > needed*1.1, "Espace /kaggle/tmp insuffisant"

    coarse_mm = np.memmap(coarse_path, dtype=np.uint8, mode='w+',
                          shape=(n_coarse, COARSE_SIZE, COARSE_SIZE))
    fine_mm   = np.memmap(fine_path, dtype=np.uint8, mode='w+',
                          shape=(n_fine, FINE_SIZE, FINE_SIZE))

    coords_fine_sorted = coords_fine_df.sort_values(
        ['id_coarse', 'fine_rank']).reset_index(drop=True)

    cur_uid, cur_arr = None, None
    for i, row in tqdm(coords_coarse_df.iterrows(), total=n_coarse, desc="Cache coarse"):
        uid = row['name']; z, yc, xc = int(row['z']), int(row['y_coarse']), int(row['x_coarse'])
        if uid != cur_uid:
            arr_norm, _, _, _ = load_ct_seg(UID_INDEX[uid])
            cur_arr = (arr_norm*255).astype(np.uint8); cur_uid = uid
        coarse_mm[i] = cur_arr[z, yc:yc+COARSE_SIZE, xc:xc+COARSE_SIZE]
    coarse_mm.flush()

    coarse_lookup = coords_coarse_df.set_index('id_coarse')[['name', 'z']]
    cur_uid, cur_arr = None, None
    for i, row in tqdm(coords_fine_sorted.iterrows(), total=n_fine, desc="Cache fine"):
        idc = int(row['id_coarse']); yf, xf = int(row['y_fine']), int(row['x_fine'])
        uid = coarse_lookup.loc[idc, 'name']; z = int(coarse_lookup.loc[idc, 'z'])
        if uid != cur_uid:
            arr_norm, _, _, _ = load_ct_seg(UID_INDEX[uid])
            cur_arr = (arr_norm*255).astype(np.uint8); cur_uid = uid
        fine_mm[i] = cur_arr[z, yf:yf+FINE_SIZE, xf:xf+FINE_SIZE]
    fine_mm.flush()

    print(f"Cache sauvegardé : coarse {os.path.getsize(coarse_path)/1e9:.2f} GB, "
          f"fine {os.path.getsize(fine_path)/1e9:.2f} GB")
    return (np.memmap(coarse_path, dtype=np.uint8, mode='r',
                      shape=(n_coarse, COARSE_SIZE, COARSE_SIZE)),
            np.memmap(fine_path, dtype=np.uint8, mode='r',
                      shape=(n_fine, FINE_SIZE, FINE_SIZE)))

In [ ]:
# ============================================================
# STEP 7 — Extraction features hiérarchiques -> HDF5
# Split par split, nettoyage immédiat du cache.
# ============================================================
BATCH_ENCODE = 512

def extract_hierarchical_split(coords_coarse_df, coords_fine_df, bounds_df,
                               coarse_mm, fine_mm, out_fname, split_name):
    print(f"\n{'='*50}\nExtraction : {split_name}")
    coords_fine_sorted = coords_fine_df.sort_values(
        ['id_coarse', 'fine_rank']).reset_index(drop=True)
    out_path = os.path.join(HDF5_DIR, out_fname)

    with h5py.File(out_path, 'w') as hf:
        for _, row in tqdm(bounds_df.iterrows(), total=len(bounds_df),
                           desc=f"Scans {split_name}"):
            scan = row['names']; start_id = int(row['start_id']); end_id = int(row['end_id'])
            n_zones = end_id - start_id + 1
            label   = UID_INDEX[scan]['label']

            scan_coarse_df = coords_coarse_df.iloc[start_id:end_id+1].reset_index(drop=True)
            fine_start = start_id * N_FPZ; fine_end = (end_id+1) * N_FPZ
            scan_fine_df = coords_fine_sorted.iloc[fine_start:fine_end].reset_index(drop=True)
            assert len(scan_fine_df) == n_zones * N_FPZ

            cf = []
            for b in range(0, n_zones, BATCH_ENCODE):
                e = min(b+BATCH_ENCODE, n_zones)
                bn = coarse_mm[start_id+b:start_id+e].copy()
                bt = torch.from_numpy(bn.astype(np.float32)/255.0).unsqueeze(1).to(DEVICE)
                with torch.no_grad(): cf.append(encoder(bt).flatten(1).cpu().numpy())
            coarse_feats = np.concatenate(cf, axis=0)

            n_fine_total = n_zones * N_FPZ; ff = []
            for b in range(0, n_fine_total, BATCH_ENCODE):
                e = min(b+BATCH_ENCODE, n_fine_total)
                bn = fine_mm[fine_start+b:fine_start+e].copy()
                bt = torch.from_numpy(bn.astype(np.float32)/255.0).unsqueeze(1).to(DEVICE)
                with torch.no_grad(): ff.append(encoder(bt).flatten(1).cpu().numpy())
            fine_feats = np.concatenate(ff, axis=0).reshape(n_zones, N_FPZ, -1)

            coarse_yx = scan_coarse_df[['y_coarse', 'x_coarse']].values.astype(np.int32)
            z_arr     = scan_coarse_df['z'].values.astype(np.int32)
            fine_yx   = scan_fine_df[['y_fine', 'x_fine']].values.astype(np.int32).reshape(n_zones, N_FPZ, 2)

            g = hf.create_group(scan)
            g.create_dataset('coarse',    data=coarse_feats, dtype=np.float32)
            g.create_dataset('fine',      data=fine_feats,   dtype=np.float32)
            g.create_dataset('coarse_yx', data=coarse_yx,    dtype=np.int32)
            g.create_dataset('fine_yx',   data=fine_yx,      dtype=np.int32)
            g.create_dataset('z',         data=z_arr,        dtype=np.int32)
            g.attrs['label']   = label
            g.attrs['n_zones'] = n_zones

    print(f"Sauvegardé : {out_path}  ({os.path.getsize(out_path)/1e9:.2f} GB)")
    return out_path

def process_split_end_to_end(coords_coarse_df, coords_fine_df, bounds_df,
                             cache_prefix, out_fname, split_name):
    coarse_mm, fine_mm = build_patch_cache(coords_coarse_df, coords_fine_df, cache_prefix)
    h5_path = extract_hierarchical_split(coords_coarse_df, coords_fine_df, bounds_df,
                                         coarse_mm, fine_mm, out_fname, split_name)
    with h5py.File(h5_path, 'r') as hf:
        keys = list(hf.keys())
        assert len(keys) == len(bounds_df)
        assert hf[keys[0]]['fine'].shape[1] == N_FPZ
        print(f"  Vérifié : {len(keys)} scans, n_fpz={hf[keys[0]]['fine'].shape[1]}")
    del coarse_mm, fine_mm
    for suf in ['_coarse.npy', '_fine.npy']:
        p = f'{cache_prefix}{suf}'
        if os.path.exists(p): os.remove(p); print(f"  Cache supprimé : {p}")
    total, used, free = shutil.disk_usage('/kaggle/tmp')
    print(f"  /kaggle/tmp libre après nettoyage : {free/1e9:.1f} GB")
    return h5_path

train_h5 = process_split_end_to_end(
    coords_coarse_train, coords_fine_train, bounds_train,
    cache_prefix=f'{CACHE_DIR}/patches_train_fold{TEST_FOLD}',
    out_fname=f'hmsips_features_train_fold{TEST_FOLD}.h5', split_name='TRAIN')

test_h5 = process_split_end_to_end(
    coords_coarse_test, coords_fine_test, bounds_test,
    cache_prefix=f'{CACHE_DIR}/patches_test_fold{TEST_FOLD}',
    out_fname=f'hmsips_features_test_fold{TEST_FOLD}.h5', split_name='TEST')

print(f"\nTrain HDF5 : {train_h5}")
print(f"Test HDF5  : {test_h5}")

In [ ]:
# ============================================================
# STEP 8 — EDA des features produites + vérification finale
# ============================================================
def summarize_h5(path):
    nz, labels, cnorm, fnorm = [], [], [], []
    with h5py.File(path, 'r') as hf:
        keys = list(hf.keys())
        for k in keys:
            g = hf[k]
            nz.append(int(g.attrs['n_zones'])); labels.append(int(g.attrs['label']))
        # stats de features sur un échantillon de 5 scans
        for k in keys[:5]:
            g = hf[k]
            cnorm.append(np.linalg.norm(g['coarse'][:], axis=1).mean())
            fnorm.append(np.linalg.norm(g['fine'][:].reshape(-1, g['fine'].shape[-1]), axis=1).mean())
    return dict(n=len(keys), nz=np.array(nz), labels=np.array(labels),
                coarse_dim=g['coarse'].shape[-1],
                cnorm=float(np.mean(cnorm)), fnorm=float(np.mean(fnorm)))

for name, path in [('TRAIN', train_h5), ('TEST', test_h5)]:
    s = summarize_h5(path)
    print(f"\n=== {name} ===")
    print(f"  scans        : {s['n']}  (pos={s['labels'].sum()}, neg={(s['labels']==0).sum()})")
    print(f"  feature dim  : {s['coarse_dim']}")
    print(f"  zones/scan   : min={s['nz'].min()} max={s['nz'].max()} moy={s['nz'].mean():.0f}")
    print(f"  patches fins/scan (=zones×{N_FPZ}) : moy={s['nz'].mean()*N_FPZ:.0f}")
    print(f"  tokens MS-IPS/scan (coarse+fine)  : moy={s['nz'].mean()*(N_FPZ+1):.0f}")
    print(f"  ||coarse||~{s['cnorm']:.2f}   ||fine||~{s['fnorm']:.2f}")

# Histogramme zones/scan
s_tr = summarize_h5(train_h5)
fig, ax = plt.subplots(figsize=(8, 4))
ax.hist(s_tr['nz'], bins=40, color='steelblue', alpha=0.8)
ax.set_xlabel('Zones grossières par scan'); ax.set_ylabel('Nb scans')
ax.set_title('Distribution des zones grossières (train)')
fig.tight_layout()
fig.savefig(f'{EDA_DIR}/zones_per_scan_train.png', dpi=110)
plt.close(fig)
print(f"\nFigure EDA : {EDA_DIR}/zones_per_scan_train.png")

print("\nFichiers livrables :")
for root, _, files in os.walk(HDF5_DIR):
    for f in sorted(files):
        p = os.path.join(root, f)
        print(f"  {p}  ({os.path.getsize(p)/1e9:.2f} GB)")